In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import requests
import joblib

import numpy
import pandas
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder

In [2]:
from lightgbm import LGBMRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import AdaBoostRegressor

In [3]:
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import scipy.stats as stats
import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [4]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [5]:
encoder = OneHotEncoder(sparse_output=False)

In [6]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]]=train[i[0]]
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1)))
    else:
        test[i[0]]=numpy.array(test[i[0]])

In [7]:
train = train.fillna(0)
test = test.fillna(0)

train_y = train['efs']
train_x = train.drop(columns=['efs'])

for i in test:
    test[i]=test[i].to_numpy()

In [8]:
alphas_alt = [14.5, 14.6, 14.7, 14.8, 14.9, 15, 15.1, 15.2, 15.3, 15.4, 15.5]
alphas2 = [5e-05, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008]
e_alphas = [0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007]
e_l1ratio = [0.8, 0.85, 0.9, 0.95, 0.99, 1]

In [9]:
ridge = make_pipeline(RobustScaler(), RidgeCV(alphas=alphas_alt))
lasso = make_pipeline(RobustScaler(), LassoCV(max_iter=1000, alphas=alphas2, random_state=42))
elasticnet = make_pipeline(RobustScaler(), ElasticNetCV(max_iter=1000, alphas=e_alphas, l1_ratio=e_l1ratio))                                
svr = make_pipeline(RobustScaler(), SVR(C= 20, epsilon= 0.008, gamma=0.0003))

In [10]:
gbr = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.05, max_depth=4, max_features='sqrt', 
                                min_samples_leaf=15, min_samples_split=10, loss='huber', random_state =42)              

In [11]:
lightgbm = LGBMRegressor(objective='regression', 
                                       num_leaves=4,
                                       learning_rate=0.01, 
                                       n_estimators=5000,
                                       max_bin=200, 
                                       verbose=-1,
                                       )

xgboost = XGBRegressor(learning_rate=0.01,n_estimators=3460,
                                     max_depth=3, min_child_weight=0,
                                     gamma=0, subsample=0.7,
                                     colsample_bytree=0.7,
                                     objective='reg:linear', nthread=-1,
                                     scale_pos_weight=1, seed=27,
                                     reg_alpha=0.00006)

model = StackingRegressor(estimators=[('ridge',ridge), ('lasso',lasso), ('elasticnet',elasticnet), ('gbr',gbr), 
                                      ('xgboost',xgboost), ('lightgbm',lightgbm)],
                                final_estimator=xgboost)

In [12]:
model.fit(train_x, train_y)

/opt/conda/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [06:08:05] WARNING: /workspace/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)
/opt/conda/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [06:14:26] WARNING: /workspace/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)
/opt/conda/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [06:14:35] WARNING: /workspace/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)
/opt/conda/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [06:14:45] WARNING: /workspace/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)
/opt/conda/lib/python3.10/site-packages/xgboost/core.py:

StackingRegressor(estimators=[('ridge',
                               Pipeline(steps=[('robustscaler', RobustScaler()),
                                               ('ridgecv',
                                                RidgeCV(alphas=[14.5, 14.6,
                                                                14.7, 14.8,
                                                                14.9, 15, 15.1,
                                                                15.2, 15.3,
                                                                15.4,
                                                                15.5]))])),
                              ('lasso',
                               Pipeline(steps=[('robustscaler', RobustScaler()),
                                               ('lassocv',
                                                LassoCV(alphas=[5e-05, 0.0001,
                                                                0.0002, 0.0003,
                                                                0.0004, 0.0005,
                                                                0.0006, 0.0007,
                                                                0.0008],
                                                        random_state=42))])),
                              ('elastic...
                                               feature_types=None, gamma=0,
                                               grow_policy=None,
                                               importance_type=None,
                                               interaction_constraints=None,
                                               learning_rate=0.01, max_bin=None,
                                               max_cat_threshold=None,
                                               max_cat_to_onehot=None,
                                               max_delta_step=None, max_depth=3,
                                               max_leaves=None,
                                               min_child_weight=0, missing=nan,
                                               monotone_constraints=None,
                                               multi_strategy=None,
                                               n_estimators=3460, n_jobs=None,
                                               nthread=-1,
                                               num_parallel_tree=None, ...))

joblib.dump(automl, 'model.pkl')

In [13]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']

In [14]:

test_predictions = model.predict(test)

In [15]:
test_predictions

array([0.16900158, 0.77455896, 0.21914704], dtype=float32)

In [16]:
submission = pandas.DataFrame({
    'id': id.values,
    'sales_hat': test_predictions,
})
submission

,id,sales_hat
0,28800,0.169002
1,28801,0.774559
2,28802,0.219147


In [17]:
submission.to_csv('submission.csv', index = False)